# Spoke Fourier Analysis

중앙에서 바깥 방향으로 좁은 각도 영역에 길게 뻗는 spoke 형태 불량을 찾기 위한 Fourier 분석 notebook입니다.

입력 raw 데이터에는 아래 역할을 하는 칼럼이 필요합니다. `.csv`와 `.txt`를 모두 읽을 수 있고, comma와 tab separator를 모두 시도합니다. 실제 파일의 칼럼명이 다르면 `Column Mapping` 입력값에서 매핑할 수 있습니다.

```text
root lot id, wafer id, chip x position, chip y position, bin number
```

`DEFECT_BIN_NOS`에 입력한 bin 값만 defect로 봅니다. 나머지 bin도 버리지 않고 wafer 전체 map을 반지름 1인 원으로 정규화하는 데 사용합니다. 별도의 x/y pitch 입력은 받지 않고, 각 wafer의 occupied coordinate extent를 이용해 x/y aspect를 자동 보정합니다. raw data에 다른 칼럼이 많아도 매핑에 필요한 칼럼만 읽습니다.

선택된 raw 칼럼은 먼저 문자열로 읽고, 좌표 칼럼만 계산 시점에 숫자로 변환합니다. 따라서 `wafer_id`가 `10` 같은 숫자형이거나 `W01` 같은 문자형이어도 처리할 수 있습니다.

## 계산식

입력한 defect bin set을 `B`라고 하면 chip별 defect indicator는 아래와 같습니다.

$$
d_i = \mathbf{1}(bin\_no_i \in B)
$$

모든 반경 영역을 사용하고, theta 방향을 기본 360개 bin으로 나눠 angular defect-rate signal을 만듭니다.

$$
p(\theta_j) = \mathrm{mean}\{d_i \mid \theta_i \in \mathrm{bin}_j\}
$$

평균 offset, 즉 전체 defect-rate 성분을 제거한 뒤 harmonic `k`의 Fourier amplitude를 계산합니다.

$$
A_k = 2\left|\frac{1}{N}\sum_j (p(\theta_j)-\bar{p})e^{-ik\theta_j}\right|
$$

spoke 불량은 특정 angle 위치가 정해져 있지 않고 좁은 angular feature로 나타날 수 있으므로, 하나의 harmonic을 고정하지 않고 high-frequency band 전체의 에너지를 score로 씁니다.

$$
\mathrm{high\_freq\_fourier\_signal} = \sqrt{\frac{1}{2}\sum_{k=k_{min}}^{k_{max}} A_k^2}
$$

`signal_to_noise`는 내부 참고용으로만 계산합니다.

$$
\mathrm{signal\_to\_noise} = \frac{\mathrm{high\_freq\_fourier\_signal}}{\mathrm{std}(p(\theta)) + 10^{-12}}
$$

In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy==1.26.4",
    "polars": "polars==1.14.0",
    "matplotlib": "matplotlib==3.9.2",
}

missing = [spec for module, spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages are already installed.")

## User Inputs

`DEFECT_BIN_NOS`는 list 또는 scalar로 입력할 수 있습니다. 예를 들어 `DEFECT_BIN_NOS = 12`는 자동으로 `[12]`처럼 처리됩니다. `Column Mapping` 값에는 raw file 안의 실제 칼럼명을 넣습니다. 검증용 wafer 선택값은 분석 실행 후의 `Selected Wafer Validation` 셀에서 별도로 입력합니다.

In [ ]:
from pathlib import Path

INPUT_FILE = Path("input.txt")
OUTPUT_CSV = Path("spoke_fourier_output.csv")

# Column Mapping: raw file 안의 실제 칼럼명을 지정합니다.
ROOT_LOT_ID_COL = "root_lot_id"
WAFER_ID_COL = "wafer_id"
CHIP_X_COL = "chip_x_pos"
CHIP_Y_COL = "chip_y_pos"
BIN_NO_COL = "bin_no"

# list 또는 scalar 모두 허용합니다. 예: [12, 13] 또는 12
DEFECT_BIN_NOS = [12]

ANGULAR_BINS = 360
HIGH_FREQ_MIN_HARMONIC = 8
HIGH_FREQ_MAX_HARMONIC = None
MIN_CHIPS = 20

## Run

아래 셀은 `spoke_fourier.py`의 backend 로직을 실행합니다. 결과는 `high_freq_fourier_signal` 기준으로 정렬됩니다.

In [ ]:
import importlib
import spoke_fourier

importlib.reload(spoke_fourier)
from spoke_fourier import SpokeConfig, run_spoke_fourier

config = SpokeConfig(
    input_path=INPUT_FILE,
    defect_bin_nos=DEFECT_BIN_NOS,
    output_csv=OUTPUT_CSV,
    group_cols=(ROOT_LOT_ID_COL, WAFER_ID_COL),
    x_col=CHIP_X_COL,
    y_col=CHIP_Y_COL,
    bin_col=BIN_NO_COL,
    angular_bins=ANGULAR_BINS,
    high_freq_min_harmonic=HIGH_FREQ_MIN_HARMONIC,
    high_freq_max_harmonic=HIGH_FREQ_MAX_HARMONIC,
    min_chips=MIN_CHIPS,
)

run = run_spoke_fourier(config, write_csv=True)
result = run.result
analysis_df = run.analysis_df

print(f"defect bin_no values: {run.defect_bin_nos}")
print(f"saved: {OUTPUT_CSV}")
print(f"result rows={result.height:,}, columns={len(result.columns):,}")
print(f"analysis_df rows={analysis_df.height:,}, columns={len(analysis_df.columns):,}")
result.head(20)

## Result Columns

- `defect_rate`: 입력한 defect bin의 wafer 전체 chip 비율입니다.
- `high_freq_fourier_signal`: high-frequency harmonic band의 Fourier energy score입니다.
- `peak_high_freq_amplitude`: high-frequency band 안에서 가장 큰 단일 harmonic amplitude입니다.
- `dominant_harmonic`: high-frequency band 안에서 amplitude가 가장 큰 harmonic입니다.
- `signal_to_noise`: 내부 참고용 정규화 score입니다.

In [ ]:
summary_cols = [
    *config.group_cols,
    "defect_rate",
    "high_freq_fourier_signal",
    "peak_high_freq_amplitude",
    "dominant_harmonic",
    "signal_to_noise",
    "theta_bins_with_chips",
    "theta_coverage",
    "total_chip_count",
    "defect_chip_count",
]
result.select([col for col in summary_cols if col in result.columns]).head(30)

## Selected Wafer Validation

전체 wafer 분석 입력과 분리된 검증용 선택값입니다. `(root_lot_id, wafer_id)` 형태로 입력합니다. wafer id는 문자열 기준으로 비교하므로 `10`과 `"10"`은 같은 값으로 처리되고, 파일 값이 `W01`이면 `"W01"`로 입력합니다.

In [ ]:
# 검증할 wafer 하나를 선택합니다. 예: ("ABCDE", 10), ("ABCDE", "W01")
WAFER_TO_PLOT = ("ABCDE", 10)

## Selected Wafer Outputs

선택한 wafer 하나에 대해 전체 chip map, theta별 defect rate, 계산 가능한 모든 harmonic amplitude를 확인합니다. `ANGULAR_BINS = 360`이면 harmonic 1부터 180까지 계산합니다.

In [ ]:
import polars as pl
from spoke_fourier import (
    build_wafer_theta_signal,
    compute_harmonic_spectrum,
    high_frequency_harmonics,
    max_calculable_harmonic,
)

theta_signal_df = build_wafer_theta_signal(
    analysis_df,
    WAFER_TO_PLOT,
    group_cols=config.group_cols,
    angular_bins=ANGULAR_BINS,
)

harmonic_spectrum_df = compute_harmonic_spectrum(
    theta_signal_df,
    angular_bins=ANGULAR_BINS,
    max_harmonic=max_calculable_harmonic(ANGULAR_BINS),
)

high_harmonics = high_frequency_harmonics(
    angular_bins=ANGULAR_BINS,
    min_harmonic=HIGH_FREQ_MIN_HARMONIC,
    max_harmonic=HIGH_FREQ_MAX_HARMONIC,
)

selected_condition = pl.lit(True)
for column, value in zip(config.group_cols, WAFER_TO_PLOT):
    selected_condition = selected_condition & (
        pl.col(column).cast(pl.Utf8, strict=False).str.strip_chars() == str(value).strip()
    )
selected_result = result.filter(selected_condition)

print(f"selected wafer: {WAFER_TO_PLOT}")
print(f"theta bins with chips: {theta_signal_df.height:,}")
selected_result

## Selected Wafer Map

선택한 wafer의 전체 chip map입니다. `DEFECT_BIN_NOS`에 해당하는 chip은 검은색, 나머지 chip은 흰색으로 표시합니다. chip 좌표에서 x/y pitch를 추정해 직사각형들이 빈 공간 없이 맞닿도록 그립니다.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Circle, Patch, Rectangle
from spoke_fourier import build_wafer_map_data

wafer_map = build_wafer_map_data(
    analysis_df,
    WAFER_TO_PLOT,
    group_cols=config.group_cols,
    x_col=config.x_col,
    y_col=config.y_col,
)
wafer_map_df = wafer_map.cells

rectangles = [
    Rectangle(
        (map_x - wafer_map.chip_width / 2, map_y - wafer_map.chip_height / 2),
        wafer_map.chip_width,
        wafer_map.chip_height,
    )
    for map_x, map_y in wafer_map_df.select("map_x", "map_y").iter_rows()
]
facecolors = [
    "black" if is_defect else "white"
    for is_defect in wafer_map_df.get_column("is_defect").to_list()
]

fig, ax = plt.subplots(figsize=(8.5, 7))
chip_collection = PatchCollection(
    rectangles,
    facecolors=facecolors,
    edgecolors="#b8b8b8",
    linewidths=0.2,
    antialiased=False,
)
ax.add_collection(chip_collection)
ax.add_patch(Circle((0, 0), wafer_map.wafer_radius, fill=False, edgecolor="black", linewidth=1.0))

plot_limit = wafer_map.wafer_radius * 1.04
ax.set_xlim(-plot_limit, plot_limit)
ax.set_ylim(-plot_limit, plot_limit)
ax.set_aspect("equal", adjustable="box")
ax.set_title(f"Wafer map: {WAFER_TO_PLOT}")
ax.axis("off")
ax.legend(
    handles=[
        Patch(facecolor="black", edgecolor="black", label=f"selected bin: {list(run.defect_bin_nos)}"),
        Patch(facecolor="white", edgecolor="#888888", label="other bins"),
    ],
    loc="upper left",
    bbox_to_anchor=(1.01, 1.0),
    frameon=False,
)
fig.tight_layout()
plt.show()

wafer_map_df

In [ ]:
import matplotlib.pyplot as plt

dominant_harmonic = None
if selected_result.height:
    dominant_harmonic = selected_result.get_column("dominant_harmonic").item()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(
    theta_signal_df.get_column("theta_rad"),
    theta_signal_df.get_column("defect_rate"),
    marker="o",
    markersize=2,
    linewidth=1,
)
ax.set_title(f"Defect rate vs theta: {WAFER_TO_PLOT}")
ax.set_xlabel("theta [rad]")
ax.set_ylabel("defect rate in theta bin")
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(
    harmonic_spectrum_df.get_column("harmonic"),
    harmonic_spectrum_df.get_column("amplitude"),
    width=0.9,
)
ax.axvspan(high_harmonics[0], high_harmonics[-1], color="orange", alpha=0.15, label="high-frequency band")
if dominant_harmonic is not None:
    ax.axvline(dominant_harmonic, color="red", linestyle="--", linewidth=1, label=f"dominant {dominant_harmonic}")
ax.set_title(f"Fourier amplitude spectrum: {WAFER_TO_PLOT}")
ax.set_xlabel("반복차수 (한 바퀴당 반복 횟수)")
ax.set_ylabel("amplitude")
ax.set_xlim(0, max_calculable_harmonic(ANGULAR_BINS) + 1)
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.show()

harmonic_spectrum_df.head(30)